# Chapter 5: Where the API key goes

This notebook runs the statements in `key-demos.cypher`, one per cell, so you can see each result as you go. The text above each cell is the comment from the script.

- Run the cells one at a time, from the top.
- Needs the 500 transactions loaded in chapter 2 and TYPESAFE_API_KEY set.

## Prerequisites

Set the following environment variables before launching Jupyter. The URI is the local Neo4j Desktop one:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

You also need:

- The plugin JAR in the `plugins` folder of your DBMS, with `dbms.security.procedures.allowlist=jev.*` in `neo4j.conf`.
- For the hosted API only, `server.jvm.additional=-DTYPESAFE_API_KEY=<your-api-key>` in `neo4j.conf`. The notebook never sees the key.

## 1. Install Dependencies

In [1]:
%pip install neo4j==5.28.1 \
             pandas==2.2.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import os
import pandas as pd

from IPython.display import display
from neo4j import GraphDatabase

## 3. Configuration

The connection details come from environment variables set before Jupyter was launched.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


## 4. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Helper

`run_cypher` runs one statement. A single map, such as a `jev.decide` result, prints as JSON. Anything else shows as a table.

In [5]:
def run_cypher(query):
    """Run one statement and show the result. A single map prints as JSON, anything
    else as a table. Statements that change the graph also report what changed."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        records = [r.data() for r in result]
        summary = result.consume()

    if len(records) == 1 and len(records[0]) == 1 and isinstance(next(iter(records[0].values())), dict):
        print(json.dumps(next(iter(records[0].values())), indent=2, default=str))
    elif records:
        display(pd.DataFrame(records))
    else:
        print("No results.")

    changes = {name: getattr(summary.counters, name, 0) for name in (
        "nodes_created", "nodes_deleted", "relationships_created",
        "relationships_deleted", "properties_set", "constraints_added",
        "constraints_removed")}
    changes = {k: v for k, v in changes.items() if v}
    if changes:
        print("Changes:", ", ".join(f"{k} {v}" for k, v in changes.items()))
    ms = (summary.result_available_after or 0) + (summary.result_consumed_after or 0)
    print(f"Completed after {ms:,} ms")

## 6. The statements

### 1. A normal call to the real endpoint. The key is sent, so metadata.authenticated is true.

In [6]:
run_cypher("""
MATCH (t:Transaction {txn_id: 'T0001'})
RETURN jev.decide(
  t {.amount, .merchant_category, .distance_from_home_km, .hour_of_day,
     .account_age_days, .prior_flags, .txns_last_24h, .typical_monthly_spend},
  {decision: {
     type: 'choice',
     instructions: 'Based on all available signals, should this transaction be flagged for review or passed through?',
     criteria: {
       Flag: 'Potential fraud. Send for review.',
       Pass: 'Likely legitimate. Process normally.'
     }
  }}
) AS result
""")

{
  "answers": {
    "decision": {
      "type": "choice",
      "choice": "Pass",
      "probabilities": {
        "Pass": 0.91,
        "Flag": 0.09
      },
      "confidence": 0.82
    }
  },
  "error_message": null,
  "metadata": {
    "authenticated": true,
    "model": "jev-latest",
    "latency_ms": 312,
    "attempts": 1
  }
}
Completed after 508 ms


### 2. A different endpoint: nothing listens on port 9 of your own machine, so the connection is refused. The error names the host and port, one attempt is made and the key is not sent, so authenticated is false.

In [7]:
run_cypher("""
RETURN jev.decide(
  {amount: 120.5, merchant_category: 'grocery'},
  {decision: {
     type: 'choice',
     instructions: 'Should this transaction be flagged or passed?',
     criteria: {Flag: 'Potential fraud.', Pass: 'Likely legitimate.'}
  }},
  {url: 'http://127.0.0.1:9/v1/systemone', max_retries: 0}
) AS result
""")

{
  "answers": null,
  "error_message": "io: ConnectException reaching 127.0.0.1:9 (after 1 attempt)",
  "metadata": {
    "authenticated": false,
    "model": "jev-latest",
    "latency_ms": 0,
    "attempts": 1
  }
}
Completed after 56 ms


### 3. The same call with the default retries: three attempts with waits of 0.5 s and 1 s between them, so it takes at least 1.5 s.

In [8]:
run_cypher("""
RETURN jev.decide(
  {amount: 120.5, merchant_category: 'grocery'},
  {decision: {
     type: 'choice',
     instructions: 'Should this transaction be flagged or passed?',
     criteria: {Flag: 'Potential fraud.', Pass: 'Likely legitimate.'}
  }},
  {url: 'http://127.0.0.1:9/v1/systemone'}
) AS result
""")

{
  "answers": null,
  "error_message": "io: ConnectException reaching 127.0.0.1:9 (after 3 attempts)",
  "metadata": {
    "authenticated": false,
    "model": "jev-latest",
    "latency_ms": 1515,
    "attempts": 3
  }
}
Completed after 1,546 ms


## 7. Teardown

In [9]:
driver.close()
print("Connection closed.")

Connection closed.
